# Reference Card Extraction

This notebook is a thin orchestrator. All extraction and visualization logic lives in
`src/create_reference_cards/`. Tune `CFG` and rerun the lower cells.

Compliance comments (R1-R4 from project/AGENT.md):
- R1: only challenge data from this workspace is used.
- R2: no pretrained weights or external models are downloaded.
- R3: this notebook does not train a parametric model.
- R4: outputs are written under `project/training_data/training_images/reference_cards/`.

In [ ]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd().parent.parent / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


from src.create_reference_cards import (
    ReferenceCardsPipelineConfig,
    initialize_reference_cards_pipeline,
    plot_crop_previews,
    plot_pipeline_steps,
    run_extraction,
    run_preview,
)

# Most-used quick knobs
IMAGES_TO_PROCESS = None
PREVIEW_IMAGE_NAME = "L1000768"
ADAPTIVE_BLOCK_SIZE = 41
ADAPTIVE_C = 5
PRE_BLUR_SIZE = 3
DILATE_SIZE_1 = 7
CLEAN_MASK_MIN_AREA_ABS = 1500
HYPOTHESIS_MIN_SCORE = 0.30
HYPOTHESIS_NMS_IOU = 0.35
FIXED_CARD_SIZE = (375, 580)
ROUNDED_CORNER_RATIO = 0.08

CFG = ReferenceCardsPipelineConfig(
    image_components={
        "L1000765": 12,
        "L1000766": 12,
        "L1000767": 14,
        "L1000768": 16,
    },
    images_to_process=IMAGES_TO_PROCESS,
    preview_image_name=PREVIEW_IMAGE_NAME,
    adaptive_block_size=ADAPTIVE_BLOCK_SIZE,
    adaptive_c=ADAPTIVE_C,
    pre_blur_size=PRE_BLUR_SIZE,
    dilate_size_1=DILATE_SIZE_1,
    clean_mask_min_area_abs=CLEAN_MASK_MIN_AREA_ABS,
    filter_card_width=372,
    filter_card_height=561,
    hypothesis_center_shift_fraction=0.07,
    hypothesis_angle_jitter_deg=2.0,
    hypothesis_angle_jitter_step_deg=0.5,
    hypothesis_border_thickness=7,
    hypothesis_interior_weight=0.48,
    hypothesis_min_border_support=0.35,
    hypothesis_min_score=HYPOTHESIS_MIN_SCORE,
    hypothesis_nms_iou=HYPOTHESIS_NMS_IOU,
    hypothesis_max_candidates=8000,
    fixed_card_width=FIXED_CARD_SIZE[0],
    fixed_card_height=FIXED_CARD_SIZE[1],
    rounded_corner_ratio=ROUNDED_CORNER_RATIO,
)

CFG

## Initialize

Resolve project paths, validate inputs, and pick the images to process.

In [ ]:
state = initialize_reference_cards_pipeline(CFG)
state.keys()

## Preview

Run the mask pipeline on a single image to inspect the intermediate steps.

In [ ]:
state = run_preview(state)
state["preview"]["gray"].shape

In [ ]:
plot_pipeline_steps(state)

## Batch Extraction

Extract crops, masks, and components for every selected reference image.

In [ ]:
state = run_extraction(state)
{name: len(r.crops) for name, r in state["results"].items()}

In [ ]:
plot_crop_previews(state)